In [1]:
import pandas as pd
import numpy as np
from abcd_helper.config import DATA_DIR, PROCESSED_DATA_DIR

In [2]:
ols_data_path = PROCESSED_DATA_DIR / "features_ols_v04.parquet"
df_ols = pd.read_parquet(ols_data_path).set_index("convo_id")
print(df_ols.shape)
df_ols.head(3)

(10042, 29)


,resolved,num_turns,share_customer_turns_10pct,num_order_products,flow_manage_account,flow_order_issue,flow_product_defect,flow_purchase_dispute,flow_shipping_issue,flow_single_item_query,...,upfront_slot_density,slot_repetition_count,total_slots_disclosed,mandatory_slots_count,is_optimal_depth_2_or_3,action_count_delta,sop_edit_distance_norm,action_thrashing_count,premature_action_attempts,action_execution_latency_mean
convo_id,,,,,,,,,,,,,,,,,,,,,
3592,0,29,4.482759,1,0,0,1,0,0,0,...,0.0,0,4,4,0,-1,0.25,0,0,7.0
9489,0,21,4.761905,1,0,0,1,0,0,0,...,0.0,0,3,3,1,0,0.00,0,0,2.0
3695,1,22,3.636364,0,0,0,0,0,0,0,...,1.0,0,1,0,1,0,0.00,0,0,0.0


In [3]:
from data_science.models import LinearModel

logit = LinearModel(
    model_type="logit",
    cov_type="HC3",
    target_col="resolved"
)

# cross validation fit
cv_logit = logit.cross_validate(df_ols, n_splits=5)
print("Logit 5-fold OOF Metrics")
display(cv_logit["oof_metrics"])

# full fit
logit.fit(df_ols)
print("Logit Ful Fit Metrics")
display(logit.evaluation)
display(logit.coef_table.sort_values("p_value"))

Logit 5-fold OOF Metrics


{'roc_auc': 0.8441684250796277,
 'pr_auc': 0.8536048991566487,
 'brier_score': 0.15932139427338826,
 'log_loss': 0.4823621812357867,
 'accuracy': 0.767078271260705,
 'precision': 0.767236910316226,
 'recall': 0.8173784977908689,
 'f1': 0.7915143952223905,
 'specificity': 0.7078091106290673}

Logit Ful Fit Metrics


<BinaryClassificationResult: N=10042, ROC-AUC=0.847, PR-AUC=0.857, F1=0.793>

,term,coef,std_err,z_stat,p_value,sig,ci_2.5%,ci_97.5%,odds_ratio,marginal_effect_ame,vif
12,flow_troubleshoot_site,-4.833694,0.199978,-24.171172,4.472901e-129,***,-5.225643,-4.441745,0.007957,-0.763031,3.051955
11,flow_subscription_inquiry,-3.314723,0.153818,-21.549595,5.341256e-103,***,-3.616201,-3.013244,0.036344,-0.523251,2.308553
6,flow_product_defect,-3.909865,0.192470,-20.314126,9.645057e-92,***,-4.287099,-3.532630,0.020043,-0.617199,4.739833
5,flow_order_issue,-3.106871,0.158026,-19.660525,4.698417e-86,***,-3.416596,-2.797146,0.044741,-0.490441,2.762804
8,flow_shipping_issue,-3.666808,0.205181,-17.871075,1.981385e-71,***,-4.068956,-3.264660,0.025558,-0.578830,5.076526
25,sop_edit_distance_norm,-2.105974,0.121358,-17.353331,1.861323e-67,***,-2.343832,-1.868116,0.121727,-0.332442,1.588226
7,flow_purchase_dispute,-2.552533,0.159758,-15.977548,1.832115e-57,***,-2.865652,-2.239414,0.077884,-0.402935,2.596463
2,share_customer_turns_10pct,-0.654015,0.041152,-15.892523,7.139485e-57,***,-0.734672,-0.573358,0.519954,-0.103241,1.437136
24,action_count_delta,-0.677791,0.046103,-14.701612,6.294367e-49,***,-0.768152,-0.587431,0.507737,-0.106994,2.014592
23,is_optimal_depth_2_or_3,1.001395,0.076701,13.055796,5.889141e-39,***,0.851063,1.151726,2.722076,0.158077,2.344858


In [4]:
ols = LinearModel(
    model_type="ols",
    cov_type="HC3",
    target_col="resolved"
)

# cross validation fit
cv_ols = ols.cross_validate(df_ols, n_splits=5)
print("OLS 5-fold OOF Metrics")
display(cv_ols["oof_metrics"])

# full fit
ols.fit(df_ols)
print("OLS Ful Fit Metrics")
display(ols.evaluation)
display(ols.coef_table.sort_values("p_value"))

OLS 5-fold OOF Metrics


{'roc_auc': 0.842806486986413,
 'pr_auc': 0.8521067517751244,
 'brier_score': 0.16158695282947796,
 'log_loss': 0.5109957798593533,
 'accuracy': 0.7648874726150169,
 'precision': 0.7625235082920158,
 'recall': 0.821060382916053,
 'f1': 0.7907100434358656,
 'specificity': 0.6986984815618221}

OLS Ful Fit Metrics


<BinaryClassificationResult: N=10042, ROC-AUC=0.846, PR-AUC=0.855, F1=0.793>

,term,coef,std_err,t_stat,p_value,sig,ci_2.5%,ci_97.5%,vif
12,flow_troubleshoot_site,-0.618164,0.018933,-32.650422,7.903462e-234,***,-0.655271,-0.581056,3.051955
11,flow_subscription_inquiry,-0.559221,0.020269,-27.589936,1.469450e-167,***,-0.598948,-0.519495,2.308553
5,flow_order_issue,-0.500010,0.020875,-23.952723,8.655697e-127,***,-0.540924,-0.459096,2.762804
6,flow_product_defect,-0.619096,0.028119,-22.017043,1.977530e-107,***,-0.674208,-0.563984,4.739833
25,sop_edit_distance_norm,-0.360737,0.018727,-19.263415,1.089500e-82,***,-0.397441,-0.324034,1.588226
8,flow_shipping_issue,-0.597551,0.031679,-18.862637,2.313845e-79,***,-0.659641,-0.535461,5.076526
7,flow_purchase_dispute,-0.376996,0.020188,-18.674192,8.029522e-78,***,-0.416564,-0.337428,2.596463
24,action_count_delta,-0.096145,0.005552,-17.317524,3.469704e-67,***,-0.107026,-0.085263,2.014592
2,share_customer_turns_10pct,-0.104236,0.006143,-16.967571,1.427133e-64,***,-0.116277,-0.092196,1.437136
0,const,0.585498,0.038650,15.148607,7.738965e-52,***,0.509745,0.661251,NaN


In [5]:
robustness_df = pd.DataFrame({
    "Feature": ols.coef_table["term"],
    "OLS Coef (p.p.)": ols.coef_table["coef"],
    "OLS Sig": ols.coef_table["sig"],
    "Logit AME (p.p.)": logit.coef_table["marginal_effect_ame"],
    "Logit Sig": logit.coef_table["sig"],
    "Logit Odds Ratio": logit.coef_table["odds_ratio"],
    "VIF": ols.coef_table["vif"],
}).sort_values("OLS Coef (p.p.)")

robustness_df

,Feature,OLS Coef (p.p.),OLS Sig,Logit AME (p.p.),Logit Sig,Logit Odds Ratio,VIF
6,flow_product_defect,-0.619096,***,-0.617199,***,0.020043,4.739833
12,flow_troubleshoot_site,-0.618164,***,-0.763031,***,0.007957,3.051955
8,flow_shipping_issue,-0.597551,***,-0.578830,***,0.025558,5.076526
11,flow_subscription_inquiry,-0.559221,***,-0.523251,***,0.036344,2.308553
5,flow_order_issue,-0.500010,***,-0.490441,***,0.044741,2.762804
7,flow_purchase_dispute,-0.376996,***,-0.402935,***,0.077884,2.596463
25,sop_edit_distance_norm,-0.360737,***,-0.332442,***,0.121727,1.588226
4,flow_manage_account,-0.260658,***,-0.290431,***,0.158843,2.326042
9,flow_single_item_query,-0.129694,***,-0.202445,***,0.277354,4.042659
2,share_customer_turns_10pct,-0.104236,***,-0.103241,***,0.519954,1.437136


In [6]:
full_encoded_data_path = PROCESSED_DATA_DIR / "features_full_v04.parquet"
df_full = pd.read_parquet(full_encoded_data_path).set_index("convo_id")
df_full.head(3)

,resolved,num_turns,share_customer_turns_10pct,num_order_products,flow_account_access,flow_manage_account,flow_order_issue,flow_product_defect,flow_purchase_dispute,flow_shipping_issue,...,upfront_slot_density,slot_repetition_count,total_slots_disclosed,mandatory_slots_count,is_optimal_depth_2_or_3,action_count_delta,sop_edit_distance_norm,action_thrashing_count,premature_action_attempts,action_execution_latency_mean
convo_id,,,,,,,,,,,,,,,,,,,,,
3592,0,29,4.482759,1,0,0,0,1,0,0,...,0.0,0,4,4,0,-1,0.25,0,0,7.0
9489,0,21,4.761905,1,0,0,0,1,0,0,...,0.0,0,3,3,1,0,0.00,0,0,2.0
3695,1,22,3.636364,0,0,0,0,0,0,0,...,1.0,0,1,0,1,0,0.00,0,0,0.0


In [7]:
from data_science.models import XGBoostModel
xgb_model = XGBoostModel(
    max_depth=4,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    target_col="resolved",
)   

# Cross-validate XGB
cv_xgb = xgb_model.cross_validate(df_full, n_splits=5)
print("XGBoost 5-Fold OOF Metrics:")
print(cv_xgb["oof_metrics"])

# Full fit XGB
xgb_model.fit(df_full)
display(xgb_model.evaluation) 
display(xgb_model.importance_table) 

XGBoost 5-Fold OOF Metrics:
{'roc_auc': 0.9018919977701034, 'pr_auc': 0.907791249829478, 'brier_score': 0.125181297456816, 'log_loss': 0.39093875531204786, 'accuracy': 0.8256323441545509, 'precision': 0.8180404354587869, 'recall': 0.8715022091310751, 'f1': 0.843925483554684, 'specificity': 0.7715835140997831}


<BinaryClassificationResult: N=10042, ROC-AUC=0.927, PR-AUC=0.932, F1=0.869>

,feature,mean_abs_shap,shap_share_pct,direction_corr,gain,gain_share_pct,split_count,cover,mean_shap
0,is_optimal_depth_2_or_3,0.541443,10.79,0.9608,18.145727,4.46,158,617.181946,0.022357
1,sop_edit_distance_norm,0.538141,10.73,-0.9382,36.304924,8.92,337,432.542419,-0.011738
2,total_slots_disclosed,0.502842,10.03,0.8865,15.293085,3.76,360,392.769104,0.012234
3,flow_account_access,0.395521,7.89,0.9813,40.178253,9.88,116,956.168030,0.087994
4,num_turns,0.350410,6.99,0.8024,4.657888,1.14,514,442.511108,-0.002160
5,flow_troubleshoot_site,0.349014,6.96,-0.9882,73.934326,18.17,78,1139.967529,-0.113714
6,share_customer_turns_10pct,0.329879,6.58,-0.9633,5.147573,1.27,470,461.880493,-0.016779
7,action_count_delta,0.329448,6.57,-0.7944,8.676233,2.13,293,549.894653,0.053040
8,mandatory_slots_count,0.304210,6.07,0.0083,17.967272,4.42,308,422.401215,-0.016003
9,all_mandatory_slots_filled,0.162595,3.24,0.9607,21.146601,5.20,85,442.530792,-0.016509


## 4. Multi-Model Benchmark Comparison (OLS vs. Logit vs. XGBoost)

We evaluate all three modeling paradigms side-by-side using **Stratified 5-Fold Cross-Validation** ($N = 10,042$ conversations). The tabular comparison below quantifies discrimination (ROC-AUC, PR-AUC), probability calibration (Brier Score, Log-Loss), and classification performance across Out-Of-Fold (OOF) evaluations and Full Fit.

In [8]:
# Assemble comprehensive multi-model benchmark evaluation table
benchmark_comparison_df = pd.DataFrame({
    "Evaluation Dimension": [
        "Discrimination", "Discrimination", "Probability Calibration", "Probability Calibration",
        "Classification", "Classification", "Classification", "Classification", "Classification"
    ],
    "Metric (5-Fold OOF CV)": [
        "ROC-AUC", "PR-AUC", "Brier Score", "Log-Loss",
        "Accuracy", "Precision", "Recall", "F1-Score", "Specificity"
    ],
    "OLS (Linear Probability)": [
        f"{cv_ols['oof_metrics']['roc_auc']:.4f}",
        f"{cv_ols['oof_metrics']['pr_auc']:.4f}",
        f"{cv_ols['oof_metrics']['brier_score']:.4f}",
        f"{cv_ols['oof_metrics']['log_loss']:.4f}",
        f"{cv_ols['oof_metrics']['accuracy']*100:.2f}%",
        f"{cv_ols['oof_metrics']['precision']*100:.2f}%",
        f"{cv_ols['oof_metrics']['recall']*100:.2f}%",
        f"{cv_ols['oof_metrics']['f1']:.4f}",
        f"{cv_ols['oof_metrics']['specificity']*100:.2f}%",
    ],
    "Logit (Logistic Regression)": [
        f"{cv_logit['oof_metrics']['roc_auc']:.4f}",
        f"{cv_logit['oof_metrics']['pr_auc']:.4f}",
        f"{cv_logit['oof_metrics']['brier_score']:.4f}",
        f"{cv_logit['oof_metrics']['log_loss']:.4f}",
        f"{cv_logit['oof_metrics']['accuracy']*100:.2f}%",
        f"{cv_logit['oof_metrics']['precision']*100:.2f}%",
        f"{cv_logit['oof_metrics']['recall']*100:.2f}%",
        f"{cv_logit['oof_metrics']['f1']:.4f}",
        f"{cv_logit['oof_metrics']['specificity']*100:.2f}%",
    ],
    "XGBoost GBDT": [
        f"{cv_xgb['oof_metrics']['roc_auc']:.4f}",
        f"{cv_xgb['oof_metrics']['pr_auc']:.4f}",
        f"{cv_xgb['oof_metrics']['brier_score']:.4f}",
        f"{cv_xgb['oof_metrics']['log_loss']:.4f}",
        f"{cv_xgb['oof_metrics']['accuracy']*100:.2f}%",
        f"{cv_xgb['oof_metrics']['precision']*100:.2f}%",
        f"{cv_xgb['oof_metrics']['recall']*100:.2f}%",
        f"{cv_xgb['oof_metrics']['f1']:.4f}",
        f"{cv_xgb['oof_metrics']['specificity']*100:.2f}%",
    ],
    "XGBoost Delta vs Logit": [
        f"+{(cv_xgb['oof_metrics']['roc_auc'] - cv_logit['oof_metrics']['roc_auc'])*100:.2f} p.p.",
        f"+{(cv_xgb['oof_metrics']['pr_auc'] - cv_logit['oof_metrics']['pr_auc'])*100:.2f} p.p.",
        f"-{((cv_logit['oof_metrics']['brier_score'] - cv_xgb['oof_metrics']['brier_score']) / cv_logit['oof_metrics']['brier_score'])*100:.1f}% error",
        f"-{(cv_logit['oof_metrics']['log_loss'] - cv_xgb['oof_metrics']['log_loss']):.4f}",
        f"+{(cv_xgb['oof_metrics']['accuracy'] - cv_logit['oof_metrics']['accuracy'])*100:.2f} p.p.",
        f"+{(cv_xgb['oof_metrics']['precision'] - cv_logit['oof_metrics']['precision'])*100:.2f} p.p.",
        f"+{(cv_xgb['oof_metrics']['recall'] - cv_logit['oof_metrics']['recall'])*100:.2f} p.p.",
        f"+{(cv_xgb['oof_metrics']['f1'] - cv_logit['oof_metrics']['f1']):.4f}",
        f"+{(cv_xgb['oof_metrics']['specificity'] - cv_logit['oof_metrics']['specificity'])*100:.2f} p.p.",
    ]
})

display(benchmark_comparison_df)


,Evaluation Dimension,Metric (5-Fold OOF CV),OLS (Linear Probability),Logit (Logistic Regression),XGBoost GBDT,XGBoost Delta vs Logit
0,Discrimination,ROC-AUC,0.8428,0.8442,0.9019,+5.77 p.p.
1,Discrimination,PR-AUC,0.8521,0.8536,0.9078,+5.42 p.p.
2,Probability Calibration,Brier Score,0.1616,0.1593,0.1252,-21.4% error
3,Probability Calibration,Log-Loss,0.5110,0.4824,0.3909,-0.0914
4,Classification,Accuracy,76.49%,76.71%,82.56%,+5.86 p.p.
5,Classification,Precision,76.25%,76.72%,81.80%,+5.08 p.p.
6,Classification,Recall,82.11%,81.74%,87.15%,+5.41 p.p.
7,Classification,F1-Score,0.7907,0.7915,0.8439,+0.0524
8,Classification,Specificity,69.87%,70.78%,77.16%,+6.38 p.p.


## 5. TreeSHAP Non-Linear Dependence & Operational Tipping Points

Linear models assume a constant marginal impact across all values. By computing native C++ TreeSHAP values for XGBoost on all $N = 10,042$ conversations, we uncover the non-linear thresholds and operational tipping points where conversation resolution probability sharply collapses.

In [9]:
# Compute TreeSHAP values for all conversations
shap_df, base_val = xgb_model.compute_shap(df_full)
base_float = float(np.ravel(base_val)[0])
print(f"Computed TreeSHAP values matrix: {shap_df.shape}, Expected Value (base): {base_float:.4f}")

X_mat = df_full.drop(columns=["resolved"])

# 1. SOP Normalized Edit Distance Tipping Point
sop_val = X_mat["sop_edit_distance_norm"]
sop_shap = shap_df["sop_edit_distance_norm"]
sop_bins = pd.cut(sop_val, bins=[-0.01, 0.0, 0.10, 0.20, 0.35, 0.50, 1.0])
sop_tipping_df = pd.DataFrame({
    "sop_norm": sop_val,
    "shap": sop_shap,
    "resolved": df_full["resolved"]
}).groupby(sop_bins, observed=False).agg(
    Dialogues=("shap", "count"),
    Mean_TreeSHAP=("shap", lambda s: f"{s.mean():+.3f}"),
    Resolution_Rate=("resolved", lambda s: f"{s.mean()*100:.1f}%")
).reset_index()

print("\n=== SOP Edit Distance Operational Tipping Points ===")
display(sop_tipping_df)

# 2. Customer Turn Share Tipping Point
cust_val = X_mat["share_customer_turns_10pct"]
cust_shap = shap_df["share_customer_turns_10pct"]
cust_bins = pd.cut(cust_val, bins=[0.0, 3.5, 4.5, 5.0, 5.5, 6.5, 10.0], include_lowest=True)
cust_tipping_df = pd.DataFrame({
    "cust_share": cust_val,
    "shap": cust_shap,
    "resolved": df_full["resolved"]
}).groupby(cust_bins, observed=False).agg(
    Dialogues=("shap", "count"),
    Mean_TreeSHAP=("shap", lambda s: f"{s.mean():+.3f}"),
    Resolution_Rate=("resolved", lambda s: f"{s.mean()*100:.1f}%")
).reset_index()

print("\n=== Customer Turn Share Operational Tipping Points ===")
display(cust_tipping_df)


Computed TreeSHAP values matrix: (10042, 30), Expected Value (base): 0.1610

=== SOP Edit Distance Operational Tipping Points ===


,sop_edit_distance_norm,Dialogues,Mean_TreeSHAP,Resolution_Rate
0,"(-0.01, 0.0]",4971,+0.468,70.9%
1,"(0.0, 0.1]",0,+nan,nan%
2,"(0.1, 0.2]",212,+0.069,40.6%
3,"(0.2, 0.35]",2242,+0.133,53.6%
4,"(0.35, 0.5]",1194,-0.789,29.0%
5,"(0.5, 1.0]",1423,-1.274,19.3%



=== Customer Turn Share Operational Tipping Points ===


,share_customer_turns_10pct,Dialogues,Mean_TreeSHAP,Resolution_Rate
0,"(-0.001, 3.5]",2806,+0.465,62.5%
1,"(3.5, 4.5]",5149,-0.068,53.0%
2,"(4.5, 5.0]",1437,-0.428,47.0%
3,"(5.0, 5.5]",378,-0.675,45.0%
4,"(5.5, 6.5]",248,-0.918,37.9%
5,"(6.5, 10.0]",24,-1.026,45.8%


## 6. Conversational Failure Mode Archetype Segmentation

We dissect the **$N = 4,610$ unresolved and escalated conversations** ($Y = 0$) by decomposing local TreeSHAP attribution into four distinct failure archetypes:
1. **Flow-Inherent Difficulty (Domain):** Hard troubleshooting and defect flows where external roadblocks prevent resolution.
2. **Procedural SOP Breakdown (AST):** Human agent guideline deviation and out-of-order execution.
3. **Information Verification Stall (DST):** Interactions stalling out during identity/order slot verification.
4. **Customer Effort Churn (Dynamics):** Conversational fatigue and clarification loops.

In [10]:
# Decompose failure attributions for unresolved dialogues (Y = 0)
failed_mask = df_full["resolved"] == 0
failed_shap = shap_df[failed_mask].copy()
failed_X = X_mat[failed_mask].copy()

# Composite failure blame across macro channels
failed_shap["ast_blame"] = -(failed_shap["sop_edit_distance_norm"].clip(upper=0) + failed_shap["action_count_delta"].clip(upper=0))
failed_shap["churn_blame"] = -(failed_shap["share_customer_turns_10pct"].clip(upper=0))
failed_shap["dst_blame"] = -(failed_shap["total_slots_disclosed"].clip(upper=0) + failed_shap["all_mandatory_slots_filled"].clip(upper=0))
flow_cols = [c for c in failed_shap.columns if c.startswith("flow_")]
failed_shap["flow_blame"] = -failed_shap[flow_cols].clip(upper=0).sum(axis=1)

blame_cols = ["ast_blame", "churn_blame", "dst_blame", "flow_blame"]
primary_blame = failed_shap[blame_cols].idxmax(axis=1).map({
    "ast_blame": "Procedural SOP Breakdown (AST)",
    "churn_blame": "Customer Effort Churn (Dynamics)",
    "dst_blame": "Information Verification Stall (DST)",
    "flow_blame": "Flow Inherent Difficulty (Domain)"
})

archetype_dist_df = pd.DataFrame({
    "Failure Mode Archetype": primary_blame.value_counts().index,
    "Unresolved Conversations": primary_blame.value_counts().values,
    "Share of All Failures": [f"{v:.1f}%" for v in (primary_blame.value_counts().values / len(failed_shap) * 100)]
})
display(archetype_dist_df)

# Behavioral profiles across archetypes
arch_profiles_df = failed_X.assign(archetype=primary_blame).groupby("archetype").agg(
    Dialogues=("sop_edit_distance_norm", "count"),
    Mean_SOP_Deviation=("sop_edit_distance_norm", lambda s: f"{s.mean():.3f}"),
    Mean_Customer_Turn_Share=("share_customer_turns_10pct", lambda s: f"{s.mean()*10:.1f}%"),
    Pct_All_Slots_Filled=("all_mandatory_slots_filled", lambda s: f"{s.mean()*100:.1f}%"),
    Mean_Turns=("num_turns", lambda s: f"{s.mean():.1f}")
).reset_index()

print("\n=== Behavioral Profiles by Failure Archetype ===")
display(arch_profiles_df)


,Failure Mode Archetype,Unresolved Conversations,Share of All Failures
0,Flow Inherent Difficulty (Domain),2055,44.6%
1,Procedural SOP Breakdown (AST),1356,29.4%
2,Information Verification Stall (DST),857,18.6%
3,Customer Effort Churn (Dynamics),342,7.4%



=== Behavioral Profiles by Failure Archetype ===


,archetype,Dialogues,Mean_SOP_Deviation,Mean_Customer_Turn_Share,Pct_All_Slots_Filled,Mean_Turns
0,Customer Effort Churn (Dynamics),342,0.148,50.4%,94.4%,23.8
1,Flow Inherent Difficulty (Domain),2055,0.308,41.1%,77.9%,21.1
2,Information Verification Stall (DST),857,0.140,37.5%,72.1%,19.0
3,Procedural SOP Breakdown (AST),1356,0.601,39.1%,78.0%,25.0
